# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/124pritivarma6001-commits/flyrank_internship_ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [49]:
%pip -q install duckdb huggingface_hub

In [50]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...): ··········


In [51]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. Two paper findings + my methodology questions


### Finding 1: Click Capture by Position Tier

The paper reports that weighted CTR decreases as content moves from higher
search positions to deeper position tiers. The reported CTR ranges from
0.420% for the Top 3 to 0.050% for positions 50+.

**Methodology question:**  
I would check how the position tiers are defined and whether differences in
impressions, page mix, or other factors could affect the comparison. The
result should be interpreted as an observed relationship rather than a
causal effect.

### Finding 2: The Freshness Multiplier

The paper reports a 5.43:1 growth-to-decline ratio for the 31-90 day
freshness window and a separate 52x impression lift for refreshed versus
stale 365+ content.

**Methodology question:**  
I would check how freshness and growth/decline labels are defined, whether
the groups are comparable, and whether sample sizes are sufficient. The
paper itself notes that some large ratios can be unstable when one group is
small.

In [52]:
import pandas as pd

# Section 1 supporting check
paper_findings = pd.DataFrame({
    "finding": [
        "Finding 3 - Click Capture by Position Tier",
        "Finding 4 - Freshness Multiplier"
    ],
    "key_measure": [
        "Weighted CTR by position tier",
        "Growth-to-decline ratio by freshness window"
    ],
    "reported_result": [
        "0.420% Top 3 -> 0.050% Deep",
        "5.43:1 for 31-90 days"
    ]
})

paper_findings

,finding,key_measure,reported_result
0,Finding 3 - Click Capture by Position Tier,Weighted CTR by position tier,0.420% Top 3 -> 0.050% Deep
1,Finding 4 - Freshness Multiplier,Growth-to-decline ratio by freshness window,5.43:1 for 31-90 days


## 2. My model under an honest split (before/after)


The Week-5 model used a random 80/20 content-level split. For this audit, I
used a client-grouped split so that content from the same client does not
appear in both training and test sets.

I kept the same Decision Tree, features, and target definition used in
Week 5. This allows the before-and-after comparison to focus on the
validation design rather than changing the model.

The client-grouped split produced 241,240 training rows and 67,994 test
rows, with zero client overlap between the two sets.

In [53]:
# Inspect the schema of the daily performance source directly

fact_daily_schema = con.sql(f"""
    DESCRIBE SELECT *
    FROM read_parquet(
        '{REL}/fact_content_daily_performance/**/*.parquet'
    )
    LIMIT 1
""").df()

fact_daily_schema[["column_name", "column_type"]]

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


In [54]:
# Build the same content-level dataset used in Week 5,
# while retaining client_hash_id for grouped validation.

model_data = con.sql(f"""
WITH daily AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        SUM(
            CASE
                WHEN gsc_impressions > 0
                THEN gsc_avg_position * gsc_impressions
                ELSE 0
            END
        ) / NULLIF(SUM(gsc_impressions), 0) AS avg_position
    FROM {TABLES['fact_daily']}
    WHERE gsc_impressions IS NOT NULL
      AND gsc_clicks IS NOT NULL
      AND gsc_avg_position IS NOT NULL
    GROUP BY client_hash_id, content_hash_id
),
base AS (
    SELECT
        *,
        CAST(clicks AS DOUBLE) / NULLIF(impressions, 0) AS ctr
    FROM daily
    WHERE impressions > 0
)
SELECT
    client_hash_id,
    content_hash_id,
    impressions,
    clicks,
    ctr,
    avg_position,
    CASE
        WHEN impressions >= 237
         AND ctr < 0.003026
         AND avg_position <= 20
        THEN 1
        ELSE 0
    END AS opportunity
FROM base
WHERE ctr IS NOT NULL
  AND avg_position IS NOT NULL
""").df()

print("Total rows:", len(model_data))
print("Unique clients:", model_data["client_hash_id"].nunique())
print("Opportunity rate:", model_data["opportunity"].mean())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows: 309234
Unique clients: 67
Opportunity rate: 0.21589475930848484


In [55]:
from sklearn.model_selection import GroupShuffleSplit

# Features used in Week 5
features = [
    "impressions",
    "ctr",
    "avg_position"
]

X = model_data[features]
y = model_data["opportunity"]
groups = model_data["client_hash_id"]

# Client-grouped 80/20 split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

train_data = model_data.iloc[train_idx].copy()
test_data = model_data.iloc[test_idx].copy()

print("Total rows:", len(model_data))
print("Training rows:", len(train_data))
print("Test rows:", len(test_data))

print("Training clients:", train_data["client_hash_id"].nunique())
print("Test clients:", test_data["client_hash_id"].nunique())

print("Training opportunity rate:", train_data["opportunity"].mean())
print("Test opportunity rate:", test_data["opportunity"].mean())

# Confirm that no client appears in both sets
overlap = set(train_data["client_hash_id"]) & set(test_data["client_hash_id"])

print("Client overlap:", len(overlap))

Total rows: 309234
Training rows: 241240
Test rows: 67994
Training clients: 53
Test clients: 14
Training opportunity rate: 0.18290084563090697
Test opportunity rate: 0.332955849045504
Client overlap: 0


In [56]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import pandas as pd

# Same model configuration as Week 5
model_honest = DecisionTreeClassifier(
    max_depth=4,
    random_state=42,
    class_weight="balanced"
)

# Training data
X_train = train_data[features]
y_train = train_data["opportunity"]

# Held-out client test data
X_test = test_data[features]
y_test = test_data["opportunity"]

# Train
model_honest.fit(X_train, y_train)

# Predict
honest_pred = model_honest.predict(X_test)

# Metrics for honest client-grouped split
honest_metrics = {
    "Accuracy": accuracy_score(y_test, honest_pred),
    "Precision": precision_score(y_test, honest_pred, zero_division=0),
    "Recall": recall_score(y_test, honest_pred, zero_division=0),
    "F1": f1_score(y_test, honest_pred, zero_division=0)
}

honest_results = pd.DataFrame(
    [honest_metrics],
    index=["Decision Tree - Client Grouped"]
)

honest_results.round(4)

,Accuracy,Precision,Recall,F1
Decision Tree - Client Grouped,1.0,1.0,1.0,1.0


In [57]:
# Week-5 results from the random 80/20 split
week5_results = pd.DataFrame({
    "Validation": ["Week-5 Random 80/20", "Week-6 Client Grouped"],
    "Accuracy": [1.0, honest_metrics["Accuracy"]],
    "Precision": [1.0, honest_metrics["Precision"]],
    "Recall": [1.0, honest_metrics["Recall"]],
    "F1": [1.0, honest_metrics["F1"]]
})

week5_results.round(4)

,Validation,Accuracy,Precision,Recall,F1
0,Week-5 Random 80/20,1.0,1.0,1.0,1.0
1,Week-6 Client Grouped,1.0,1.0,1.0,1.0


## 3. Leakage Audit

The final model uses impressions, CTR, and average position as features.
These same variables are also used to construct the opportunity label using
the Week-4 threshold rule.

This is not future-data leakage, because no future outcome or future-window
information is used. However, it creates target construction dependence:
the model is learning to reproduce a deterministic rule based on the same
features.

The rule check produced zero label-rule mismatches. Therefore, the
near-perfect validation scores should be interpreted as successful rule
reproduction rather than independent predictive performance.

In [58]:
# Audit the relationship between the final features and the target rule

audit = pd.DataFrame({
    "feature": [
        "impressions",
        "ctr",
        "avg_position"
    ],
    "used_to_construct_label": [
        True,
        True,
        True
    ],
    "used_as_model_feature": [
        True,
        True,
        True
    ]
})

audit


,feature,used_to_construct_label,used_as_model_feature
0,impressions,True,True
1,ctr,True,True
2,avg_position,True,True


In [59]:
# Verify that the Week-5 label is reproduced exactly by its defining rule

rule_check = (
    (model_data["impressions"] >= 237)
    & (model_data["ctr"] < 0.003026)
    & (model_data["avg_position"] <= 20)
).astype(int)

print("Rows:", len(model_data))
print("Label-rule mismatches:", (rule_check != model_data["opportunity"]).sum())

Rows: 309234
Label-rule mismatches: 0


## 4. Claim Rewrite

### Original Claim:
The Decision Tree achieves 100% accuracy, precision, recall, and F1 score
and therefore provides a highly accurate model for identifying content
opportunities.

### Safer Claim:
In this dataset, the Decision Tree reproduced the existing rule-based
opportunity label with near-perfect measured performance on both random and
client-grouped validation splits.

Because the label is deterministically constructed from impressions, CTR,
and average position, the result should be treated as evidence of rule
reproduction and decision support, rather than proof of independent
predictive performance or causal impact.

### Real Failure Example:
The Week-5 model had one observed classification error, showing that
boundary cases can still produce disagreements with the rule despite the
very high aggregate metrics.

In [60]:
# Final claim-audit summary

claim_audit = pd.DataFrame({
    "item": [
        "Validation design",
        "Features",
        "Target construction",
        "Observed result",
        "Interpretation"
    ],
    "result": [
        "Random 80/20 and client-grouped split",
        "Impressions, CTR, average position",
        "Deterministic threshold rule using the same features",
        "1.0 metrics on both validation designs",
        "Rule reproduction, not independent predictive evidence"
    ]
})

claim_audit

,item,result
0,Validation design,Random 80/20 and client-grouped split
1,Features,"Impressions, CTR, average position"
2,Target construction,Deterministic threshold rule using the same fe...
3,Observed result,1.0 metrics on both validation designs
4,Interpretation,"Rule reproduction, not independent predictive ..."


In [61]:
failure_example = pd.DataFrame({
    "content_hash_id": ["content_d6934026c54bac2d"],
    "impressions": [2416.0],
    "clicks": [1.0],
    "ctr": [0.000414],
    "avg_position": [20.0],
    "actual_opportunity": [1],
    "week5_tree_prediction": [0]
})

failure_example

,content_hash_id,impressions,clicks,ctr,avg_position,actual_opportunity,week5_tree_prediction
0,content_d6934026c54bac2d,2416.0,1.0,0.000414,20.0,1,0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.